In [1]:
%cd /kaggle/working
!rm -rf g8-traffic-sign-detection
!git clone https://github.com/salmaaz29/g8-traffic-sign-detection.git
%cd g8-traffic-sign-detection
!pip install -q pycocotools
!ls src/models

/kaggle/working
Cloning into 'g8-traffic-sign-detection'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 102 (delta 24), reused 75 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 35.80 MiB | 35.08 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/kaggle/working/g8-traffic-sign-detection
base.py  frcnn_dataset.py  frcnn.py  __init__.py  train_frcnn.py  yolo.py


In [5]:
import glob, os, torch

# au cas où src/__init__.py manquerait
!touch src/__init__.py src/models/__init__.py

hits = glob.glob('/kaggle/input/**/gtsdb_4cls/coco/train.json', recursive=True)
assert hits, "Ajoute la sortie du notebook 01 en Input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
print(DATA, os.listdir(DATA + '/images'))

print("GPU :", torch.cuda.is_available())

/kaggle/input/notebooks/salmaazhich/01-kaggle-data-prepa/g8-traffic-sign-detection/data/processed/gtsdb_4cls ['test', 'train', 'val']
GPU : True


In [6]:
!python -m src.models.train_frcnn --data-dir {DATA} --epochs 2 --limit 40 --out /kaggle/working/weights_test

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|█████████████████████████████████████████| 160M/160M [00:01<00:00, 167MB/s]
epoch   1/2 | loss 0.7394 | val mAP50 0.123 | mAP50-95 0.056 | 23s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.056)
epoch   2/2 | loss 0.2802 | val mAP50 0.326 | mAP50-95 0.196 | 16s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.196)


In [8]:
!python -m src.models.train_frcnn --data-dir {DATA} --epochs 30 --out /kaggle/working/weights

epoch   1/30 | loss 0.2436 | val mAP50 0.842 | mAP50-95 0.552 | 97s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.552)
epoch   2/30 | loss 0.0895 | val mAP50 0.935 | mAP50-95 0.643 | 96s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.643)
epoch   3/30 | loss 0.0652 | val mAP50 0.965 | mAP50-95 0.778 | 97s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.778)
epoch   4/30 | loss 0.0543 | val mAP50 0.964 | mAP50-95 0.750 | 97s
epoch   5/30 | loss 0.0471 | val mAP50 0.967 | mAP50-95 0.759 | 97s
epoch   6/30 | loss 0.0446 | val mAP50 0.959 | mAP50-95 0.744 | 96s
epoch   7/30 | loss 0.0441 | val mAP50 0.967 | mAP50-95 0.721 | 97s
epoch   8/30 | loss 0.0397 | val mAP50 0.968 | mAP50-95 0.726 | 97s
epoch   9/30 | loss 0.0318 | val mAP50 0.968 | mAP50-95 0.773 | 96s
epoch  10/30 | loss 0.0311 | val mAP50 0.968 | mAP50-95 0.780 | 96s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.780)
epoch  11/30 | loss 0.0284 | val mAP50 0.966 | mAP50-95 0.796 | 96s
  -> frcnn_best.pth sauvegardé (mAP50-95 = 0.796)
ep

In [9]:
!ls -lh /kaggle/working/weights

total 317M
-rw-r--r-- 1 root root 159M Nov 18 16:34 frcnn_best.pth
-rw-r--r-- 1 root root 159M Nov 18 16:47 frcnn_last.pth


In [10]:
import cv2, glob
from src.models.frcnn import FasterRCNNDetector

det = FasterRCNNDetector('/kaggle/working/weights/frcnn_best.pth')
path = sorted(glob.glob(DATA + '/images/test/*.png'))[0]
img = cv2.imread(path)
for d in det.predict(img, conf=0.25):
    print(d)

{'box': (394.6573791503906, 511.6769714355469, 420.0272521972656, 536.6406860351562), 'score': 0.9983842372894287, 'class_id': 2, 'label': 'mandatory'}


In [11]:
!python -m src.evaluation.evaluate --model frcnn --weights /kaggle/working/weights/frcnn_best.pth --data-dir {DATA} --split test

eval: 100%|█████████████████████████████████████| 90/90 [00:52<00:00,  1.72it/s]

frcnn | prétraitement=none | dégradation=aucune niveau=0 | 90 images
  map50        : 0.988
  map50_95     : 0.820
  precision    : 0.880
  recall       : 0.980
  mean_conf    : 0.942
  ms_per_image : 212.934
  fps          : 4.696
  (vitesse mesurée à conf=0.25 ; à conf=0.001 : 213.0 ms/image)
  AP50 par classe : {'prohibitory': 1.0, 'danger': 0.987, 'mandatory': 0.987, 'other': 0.978}
